# Test Suite: Ranking Stability Analysis

Unit tests for the Alzheimer's Research GNN project.

In [ ]:
"""
Unit tests for ranking stability analysis module.
"""

import numpy as np
import pandas as pd
import pytest

from src.analysis.stability import (
    CrossCohortAnalyzer,
    PermutationTest,
    RankingStabilityAnalyzer,
)


@pytest.fixture
def sample_bootstrap_ranks():
    """Create sample bootstrap ranking data."""
    np.random.seed(42)
    proteins = [f"PROT_{i}" for i in range(100)]
    n_bootstrap = 10

    # Create bootstrap ranks (each protein has ranking across bootstrap iterations)
    bootstrap_ranks = {}
    for protein in proteins:
        # Some proteins stable (low variance in ranks), some unstable
        if proteins.index(protein) < 20:
            # Stable proteins - consistent low ranks
            ranks = [np.random.randint(0, 10) for _ in range(n_bootstrap)]
        else:
            # Unstable proteins - random ranks across iterations
            ranks = [np.random.randint(0, 100) for _ in range(n_bootstrap)]
        bootstrap_ranks[protein] = ranks

    return bootstrap_ranks, proteins


def test_ranking_stability_initialization(sample_bootstrap_ranks):
    """Test RankingStabilityAnalyzer initialization."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    assert analyzer.n_proteins == 100
    assert analyzer.n_bootstrap == 10
    assert len(analyzer.protein_ids) == 100


def test_get_top_k_proteins(sample_bootstrap_ranks):
    """Test getting top-k proteins from bootstrap iteration."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    top_20 = analyzer.get_top_k_proteins(iteration=0, k=20)

    assert isinstance(top_20, set)
    assert len(top_20) <= 20
    assert all(isinstance(p, str) for p in top_20)


def test_jaccard_similarity(sample_bootstrap_ranks):
    """Test Jaccard similarity computation."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    set1 = {"PROT_0", "PROT_1", "PROT_2"}
    set2 = {"PROT_1", "PROT_2", "PROT_3"}

    jaccard = analyzer.jaccard_similarity(set1, set2)

    assert 0 <= jaccard <= 1
    assert jaccard == 0.5  # 2 intersect / 4 union


def test_jaccard_stability(sample_bootstrap_ranks):
    """Test Jaccard stability computation."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    result = analyzer.compute_jaccard_stability(k=20)

    assert "jaccard_similarities" in result
    assert "mean_jaccard" in result
    assert "std_jaccard" in result
    assert "median_jaccard" in result
    assert 0 <= result["mean_jaccard"] <= 1


def test_kuncheva_index(sample_bootstrap_ranks):
    """Test Kuncheva stability index."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    kuncheva = analyzer.compute_kuncheva_index(k=20)

    assert isinstance(kuncheva, float)
    assert -1 <= kuncheva <= 1  # Typical range


def test_stability_curves(sample_bootstrap_ranks):
    """Test stability curves computation."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    stability_df = analyzer.compute_stability_curves(k_values=[10, 20, 30])

    assert len(stability_df) == 3
    assert "k" in stability_df.columns
    assert "mean_jaccard" in stability_df.columns
    assert "kuncheva" in stability_df.columns
    assert stability_df["k"].tolist() == [10, 20, 30]


def test_cross_cohort_analyzer():
    """Test CrossCohortAnalyzer."""
    # Create sample reports for multiple cohorts
    proteins = [f"PROT_{i}" for i in range(100)]
    cohort_reports = {
        "ROSMAP": pd.DataFrame({
            "protein": proteins[:100],
            "mean_attr": np.random.random(100),
        }).sort_values("mean_attr", ascending=False),
        "MSBB": pd.DataFrame({
            "protein": proteins[:100],
            "mean_attr": np.random.random(100),
        }).sort_values("mean_attr", ascending=False),
        "MAYO": pd.DataFrame({
            "protein": proteins[:100],
            "mean_attr": np.random.random(100),
        }).sort_values("mean_attr", ascending=False),
    }

    analyzer = CrossCohortAnalyzer(cohort_reports, "/tmp")

    assert analyzer.cohort_names == ["ROSMAP", "MSBB", "MAYO"]


def test_get_top_k_proteins_cohort():
    """Test getting top-k proteins from cohort."""
    proteins = [f"PROT_{i}" for i in range(50)]
    report = pd.DataFrame({
        "protein": proteins,
        "mean_attr": np.arange(len(proteins), 0, -1),
    })

    cohort_reports = {"TEST": report}
    analyzer = CrossCohortAnalyzer(cohort_reports, "/tmp")

    top_10 = analyzer.get_top_k_proteins("TEST", k=10)

    assert len(top_10) == 10
    assert set(top_10) == set(proteins[:10])


def test_pairwise_overlap():
    """Test pairwise overlap computation."""
    proteins = [f"PROT_{i}" for i in range(100)]

    # Create similar rankings for each cohort (some overlap expected)
    cohort_reports = {
        "ROSMAP": pd.DataFrame({
            "protein": proteins,
            "mean_attr": np.arange(len(proteins), 0, -1),
        }),
        "MSBB": pd.DataFrame({
            "protein": proteins,  # Same proteins, same order
            "mean_attr": np.arange(len(proteins), 0, -1),
        }),
    }

    analyzer = CrossCohortAnalyzer(cohort_reports, "/tmp")
    overlap = analyzer.compute_pairwise_overlap(k=20)

    assert len(overlap) == 1  # Only 1 pair
    assert "intersection" in overlap.columns
    assert "jaccard" in overlap.columns
    assert overlap.iloc[0]["intersection"] == 20  # Perfect overlap when same ranking


def test_shared_proteins():
    """Test getting shared proteins across cohorts."""
    proteins = [f"PROT_{i}" for i in range(100)]

    cohort_reports = {
        "ROSMAP": pd.DataFrame({
            "protein": proteins,
            "mean_attr": np.arange(len(proteins), 0, -1),
        }),
        "MSBB": pd.DataFrame({
            "protein": proteins,
            "mean_attr": np.arange(len(proteins), 0, -1),
        }),
        "MAYO": pd.DataFrame({
            "protein": proteins,
            "mean_attr": np.arange(len(proteins), 0, -1),
        }),
    }

    analyzer = CrossCohortAnalyzer(cohort_reports, "/tmp")
    shared = analyzer.get_shared_proteins(k=50)

    assert "all_shared" in shared
    assert "n_all_shared" in shared
    assert shared["n_all_shared"] == 50  # All same ranking


def test_permutation_test(sample_bootstrap_ranks):
    """Test permutation test."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks

    perm_test = PermutationTest(bootstrap_ranks, proteins, n_permutations=10)
    result = perm_test.run_permutation_test(k=20)

    assert "real_stability" in result
    assert "perm_stabilities" in result
    assert "p_value" in result
    assert 0 <= result["p_value"] <= 1
    assert len(result["perm_stabilities"]) == 10


def test_jaccard_edge_cases(sample_bootstrap_ranks):
    """Test Jaccard similarity edge cases."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    # Empty sets
    assert analyzer.jaccard_similarity(set(), set()) == 0.0
    assert analyzer.jaccard_similarity({"A"}, set()) == 0.0

    # Identical sets
    set1 = {"A", "B", "C"}
    assert analyzer.jaccard_similarity(set1, set1) == 1.0

    # Disjoint sets
    set1 = {"A", "B"}
    set2 = {"C", "D"}
    assert analyzer.jaccard_similarity(set1, set2) == 0.0


def test_stability_metrics_range(sample_bootstrap_ranks):
    """Test that stability metrics are in expected ranges."""
    bootstrap_ranks, proteins = sample_bootstrap_ranks
    analyzer = RankingStabilityAnalyzer(bootstrap_ranks, proteins, n_bootstrap=10)

    for k in [10, 20, 30, 40, 50]:
        jaccard_result = analyzer.compute_jaccard_stability(k)
        assert 0 <= jaccard_result["mean_jaccard"] <= 1
        assert 0 <= jaccard_result["median_jaccard"] <= 1

        kuncheva = analyzer.compute_kuncheva_index(k)
        assert -2 <= kuncheva <= 1  # Reasonable range


def test_overlap_transitivity():
    """Test that overlap computation is reasonable."""
    proteins = [f"PROT_{i}" for i in range(100)]

    # Create cohorts with known overlaps
    report1 = pd.DataFrame({
        "protein": proteins,
        "mean_attr": np.arange(len(proteins), 0, -1),
    })

    # Shift proteins slightly
    proteins_shifted = proteins[40:] + proteins[:40]
    report2 = pd.DataFrame({
        "protein": proteins_shifted,
        "mean_attr": np.arange(len(proteins), 0, -1),
    })

    cohort_reports = {"C1": report1, "C2": report2}
    analyzer = CrossCohortAnalyzer(cohort_reports, "/tmp")

    overlap = analyzer.compute_pairwise_overlap(k=50)
    jaccard = overlap.iloc[0]["jaccard"]

    # With 40-protein shift in 100-protein space, expect some overlap
    assert 0 < jaccard < 1


if __name__ == "__main__":
    pytest.main([__file__, "-v"])
